# STK-APEX 68M — 1. fázisú pretraining

**Beállítás:** Settings → Accelerator → **GPU T4 x2** (vagy P100)

A notebook magától megkeresi a korpuszt és a kódot a `/kaggle/input` alatt,
tehát a dataset elnevezése mindegy.

A session 12 óra után leáll. A `ckpt/last.pt` a `/kaggle/working`-ben marad,
az Output-ból letölthető, és a következő futásban `--resume`-mal folytatható.

In [ ]:
# 1. Környezet felderítése
import glob, os, subprocess, sys, zipfile
from pathlib import Path

print('--- /kaggle/input tartalma ---')
for p in sorted(glob.glob('/kaggle/input/*')):
    tot = sum(f.stat().st_size for f in Path(p).rglob('*') if f.is_file())
    print(f'  {p}   ({tot/1e9:.2f} GB)')
    for f in sorted(Path(p).rglob('*'))[:8]:
        if f.is_file():
            print(f'      {f.name}  {f.stat().st_size/1e6:.1f} MB')

import torch
print('\nCUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('torch:', torch.__version__)

In [ ]:
# 2. Kód kicsomagolása + korpusz megkeresése
WORK = Path('/kaggle/working')

zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
for z in zips:
    if 'stk' in Path(z).name.lower() or 'code' in Path(z).name.lower():
        with zipfile.ZipFile(z) as f:
            f.extractall(WORK)
        print('kicsomagolva:', z)

# ha a kód nem zipben, hanem mappaként van feltöltve
if not (WORK / 'stk_apex').exists():
    for cand in glob.glob('/kaggle/input/**/stk_apex', recursive=True):
        subprocess.run(['cp', '-r', str(Path(cand).parent) + '/.', str(WORK)])
        print('másolva:', cand)
        break

TOKENS = None
for c in glob.glob('/kaggle/input/**/*.bin', recursive=True):
    if 'token' in Path(c).name.lower() and Path(c).stat().st_size > 10_000_000:
        TOKENS = c
        break

os.chdir(WORK)
print('\nmunkakönyvtár :', WORK, sorted(os.listdir(WORK))[:10])
print('korpusz       :', TOKENS,
      f'({Path(TOKENS).stat().st_size/1e9:.2f} GB)' if TOKENS else 'NEM TALÁLHATÓ')
assert TOKENS, 'Nincs tokens.bin a /kaggle/input alatt — add hozzá a datasetet.'
assert (WORK / 'stk_apex').exists(), 'Nincs stk_apex kód — töltsd fel a kód-zipet.'

In [ ]:
# 3. Átbocsátás-mérés (60 s) — ez dönti el, hány óra a teljes tanítás.
#    Az SGS scan sok apró kernelt indít; ha a token/s jóval 15k alatt van,
#    a scan a szűk keresztmetszet, nem a GEMM.
!python training/train_stage1.py --benchmark --bs 16 --tokens {TOKENS}

In [ ]:
# 4. Éles tanítás. Ha a 3. cella szerint a bs=16 kevés GPU-memóriát használ,
#    emeld (bs=32/48) — a nagyobb batch jobban kihasználja a tensor core-okat.
#    Folytatáshoz:  --resume ckpt/last.pt
!python training/train_stage1.py \
    --steps 50000 --bs 16 --accum 4 --lr 6e-4 \
    --tokens {TOKENS} \
    --ckpt-dir /kaggle/working/ckpt \
    --ckpt-every 2000 --eval-every 2000

In [ ]:
# 5. Checkpoint ellenőrzése — ez marad meg Output-ként.
for f in sorted(Path('/kaggle/working/ckpt').glob('*.pt')):
    print(f'{f.name}  {f.stat().st_size/1e6:.0f} MB')